In [2]:
!pip install openai langchain langchain-huggingface langchain-qdrant qdrant-client sentence-transformers python-dotenv

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [1]:
import sys
!{sys.executable} -m pip install langchain-huggingface langchain-community langchain-qdrant qdrant-client sentence-transformers openai langchain python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ingest.py
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# 1. Cargar y dividir el documento
loader = TextLoader("intro-to-llms-karpathy.txt", encoding="utf-8")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
docs = text_splitter.split_documents(documents)
print(f"✅ {len(docs)} chunks generados")

# 2. Embeddings locales (sin API key)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# 3. Crear colección Qdrant persistente en disco
#    all-mpnet-base-v2 produce vectores de dimensión 768
COLLECTION   = "karpathy_qdrant"
PERSIST_PATH = "./db/karpathy_qdrant"

qdrant_client = QdrantClient(path=PERSIST_PATH)
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
vectorstore.add_documents(docs)
print(f"✅ Base de datos vectorial creada en {PERSIST_PATH}")

qdrant_client.close()
print("✅ Conexión cerrada correctamente")

✅ 81 chunks generados


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Base de datos vectorial creada en ./db/karpathy_qdrant
✅ Conexión cerrada correctamente


In [3]:
# rag_pipeline.py
import os
from dotenv import load_dotenv
from openai import OpenAI

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.language_models.llms import LLM
from langchain_core.documents import Document
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from qdrant_client import QdrantClient
from typing import List

load_dotenv()

# ── 1. LLM: NVIDIA NIM via cliente OpenAI-compatible ──────────
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY"),
)
MODEL = "meta/llama-3.3-70b-instruct"

class NvidiaLLM(LLM):
    """
    Wrapper mínimo sobre el cliente OpenAI apuntando a NVIDIA NIM.
    Hereda de LangChain LLM para ser compatible con LCEL ( | ).
    Mismo patrón que ChatGoogleGenerativeAI pero sin dependencias de Google.
    """
    model: str = MODEL

    @property
    def _llm_type(self) -> str:
        return "nvidia-nim"

    def _call(
        self,
        prompt: str,
        stop=None,
        run_manager: CallbackManagerForLLMRun = None,
        **kwargs,
    ) -> str:
        response = nvidia_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
        )
        return response.choices[0].message.content

llm = NvidiaLLM()
print(f"✅ LLM: NVIDIA NIM / {MODEL}")

# ── 2. Embeddings — mismo modelo que en la ingesta ─────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# ── 3. Cargar Qdrant desde disco ───────────────────────────────
COLLECTION   = "karpathy_qdrant"
PERSIST_PATH = "./db/karpathy_qdrant"

qdrant_client = QdrantClient(path=PERSIST_PATH)
vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
print("✅ Vector store cargado desde disco")

# ── 4. Retriever ───────────────────────────────────────────────
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 4, "score_threshold": 0.3},
)

# ── 5. Pipeline LCEL ───────────────────────────────────────────
def format_docs(docs: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[Fragmento {i+1}]\n{d.page_content}"
        for i, d in enumerate(docs)
    )

prompt = ChatPromptTemplate.from_template("""
Eres un asistente experto. Responde la pregunta basándote ÚNICAMENTE
en el contexto proporcionado. Si la información no está en el contexto,
di explícitamente que no la tienes.

CONTEXTO:
{context}

PREGUNTA: {question}

RESPUESTA:
""")

rag_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# ── 6. Prueba rápida ───────────────────────────────────────────
question = "¿Qué es retrieval augmented generation?"
answer   = rag_chain.invoke(question)
print(f"\nPregunta: {question}")
print(f"Respuesta: {answer.strip()}")

✅ LLM: NVIDIA NIM / meta/llama-3.3-70b-instruct


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Vector store cargado desde disco

Pregunta: ¿Qué es retrieval augmented generation?
Respuesta: La retrieval augmented generation es una técnica que permite a los modelos de lenguaje grande, como el GPT, hacer referencia a chunks de texto en archivos subidos y utilizarlos como información de referencia para generar respuestas. Esto se logra mediante la capacidad del modelo para "navegar" por los archivos subidos, en lugar de navegar por Internet, y utilizar la información contenida en ellos para generar respuestas más precisas y personalizadas.


In [9]:
# generate_answers.py
import json
# from rag_pipeline import rag_chain, retriever  # importar desde Fase B

with open("questions.json", encoding="utf-8") as f:
    test_questions = json.load(f)

print(f"📋 Respondiendo {len(test_questions)} preguntas...\n")

results = []
for i, question in enumerate(test_questions, 1):
    question = question["question"]
    print(f"  [{i}/{len(test_questions)}] {question[:70]}...")

    source_docs = retriever.invoke(question)
    answer      = rag_chain.invoke(question)

    results.append({
        "question": question,
        "answer":   answer,
        "contexts": [doc.page_content for doc in source_docs],
    })

with open("my_rag_output.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("\n✅ Resultados guardados en my_rag_output.json")

📋 Respondiendo 50 preguntas...

  [1/50] What are some security challenges associated with large language model...
  [2/50] What is the purpose of the base model in the process of developing an ...
  [3/50] What is an adversarial example in the context of large language models...
  [4/50] What are the challenges associated with large language models in the c...
  [5/50] What are the key components of large language models and how do they f...
  [6/50] What is an adversarial example in the context of large language models...
  [7/50] What is the significance of parameters and weights in the functioning ...
  [8/50] What methods does the language model use for data collection and organ...
  [9/50] What is the significance of Meta AI in the development of large langua...
  [10/50] What is the purpose of a universal transferable suffix in the context ...
  [11/50] What is the concept of jailbreaking the model in the context of large ...
  [12/50] What are some capabilities of ChatGPT in ha